In [ ]:
#install uv package manager
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = f"{os.environ['PATH']}:/root/.cargo/bin"

# Clone mjlab and install
!git clone https://github.com/mujocolab/mjlab.git
%cd mjlab
!uv sync
# 1. Mount Drive first
from google.colab import drive
drive.mount('/content/drive')

# 2. Start training
!uv run train Mjlab-Velocity-Flat-Unitree-G1 --env.scene.num-envs 4096

In [ ]:
# Create a destination folder in your Drive
!mkdir -p /content/drive/MyDrive/mjlab_final_results

# Copy everything from the Colab local disk to your permanent Drive
!cp -r /content/mjlab/mjlab/mjlab/mjlab/logs /content/drive/MyDrive/mjlab_final_results/

In [ ]:
# 1. Install headless display dependencies
!apt-get update && apt-get install -y libgl1-mesa-glx libosmesa6

# 2. Set environment variables so MuJoCo knows to use OSMesa (software rendering)
import os
os.environ['MUJOCO_GL'] = 'osmesa'
os.environ['PYOPENGL_PLATFORM'] = 'osmesa'

!uv run play Mjlab-Velocity-Flat-Unitree-G1 --help

In [ ]:
# Define the base path for your training run
BASE_PATH="/content/mjlab/mjlab/mjlab/mjlab/logs/rsl_rl/g1_velocity/2026-03-09_21-42-00"

## 1. Early Model (0)
#!timeout 60s uv run play Mjlab-Velocity-Flat-Unitree-G1 \
#  --checkpoint-file /content/mjlab/mjlab/mjlab/mjlab/logs/rsl_rl/g1_velocity/2026-03-09_21-42-00/model_0.pt \
#  --video True \
#  --video-length 100
## 2. Early Model (5k)
#!timeout 60s uv run play Mjlab-Velocity-Flat-Unitree-G1 \
#  --checkpoint-file /content/mjlab/mjlab/mjlab/mjlab/logs/rsl_rl/g1_velocity/2026-03-09_21-42-00/model_5000.pt \
#  --video True \
#  --video-length 100

## 3. Intermediate Model (15k)
#!timeout 60s uv run play Mjlab-Velocity-Flat-Unitree-G1 \
#  --checkpoint-file /content/mjlab/mjlab/mjlab/mjlab/logs/rsl_rl/g1_velocity/2026-03-09_21-42-00/model_15000.pt \
#  --video True \
#  --video-length 100

## 4. Refined Model (25k)
#!timeout 60s uv run play Mjlab-Velocity-Flat-Unitree-G1 \
#  --checkpoint-file /content/mjlab/mjlab/mjlab/mjlab/logs/rsl_rl/g1_velocity/2026-03-09_21-42-00/model_25000.pt \
#  --video True \
#  --video-length 100

# 5. Early Model (29k)
!timeout 60s uv run play Mjlab-Velocity-Flat-Unitree-G1 \
  --checkpoint-file /content/mjlab/mjlab/mjlab/mjlab/logs/rsl_rl/g1_velocity/2026-03-09_21-42-00/model_29000.pt \
  --video True \
  --video-length 100





In [ ]:
!uv run train Mjlab-Velocity-Rough-Unitree-G1 --help

In [ ]:
import torch

# 1. Path to your fix-model
checkpoint_path = "/content/mjlab/mjlab/mjlab/mjlab/logs/rsl_rl/g1_velocity/2026-03-09_21-42-00/model_29999_final_fix.pt"
checkpoint = torch.load(checkpoint_path)

# 2. REMOVE the optimizer state
# This forces the training script to create a new optimizer with the correct 286/298 shapes.
if 'optimizer_state_dict' in checkpoint:
    del checkpoint['optimizer_state_dict']
    print("Old optimizer state removed.")

# 3. Save as the "Clean" version
clean_path = "/content/mjlab/mjlab/mjlab/mjlab/logs/rsl_rl/g1_velocity/2026-03-09_21-42-00/model_29999_rough_clean.pt"
torch.save(checkpoint, clean_path)
print(f"Success! Cleaned model saved to: {clean_path}")

In [ ]:
!uv run train Mjlab-Velocity-Rough-Unitree-G1 \
  --agent.resume True \
  --agent.load-run 2026-03-09_21-42-00 \
  --agent.load-checkpoint model_29999_rough_clean.pt \
  --agent.logger wandb \
  --agent.wandb-project "G1_Phase2_Hardening" \
  --env.scene.num-envs 4096

In [ ]:
#install uv package manager
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = f"{os.environ['PATH']}:/root/.cargo/bin"

# Clone mjlab and install
!git clone https://github.com/mujocolab/mjlab.git
%cd mjlab
!uv sync
# 1. Mount Drive first
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!uv run train Mjlab-Velocity-Rough-Unitree-G1 \
  --agent.resume False \
  --agent.load-checkpoint /content/drive/MyDrive/mjlab_final_results/refined\ models/model_29999_rough_clean.pt \
  --agent.logger wandb \
  --agent.wandb-project "G1_Phase2_T4" \
  --env.scene.num-envs 2048